# Session 12: Exercises

**Before session 13.** Predict, then measure. Answers go in the **✍️** cells, and code TODOs are marked
`# TODO`.

Exercises 2's server step and the case studies make **no** model calls; the others say roughly how many
they make. Knowledge questions also embed with `gemini-embedding-2`, so you need `GEMINI_API_KEY` in
`labs/.env`, and the knowledge base built (`uv run python kb/ingest.py` from `project/`, then
`--scheme contextual` for exercise 3).

**Which notebook each one follows.** Session 12 is taught in two notebooks:

| After `session-12-multiagent.ipynb` | After `session-12-mcp.ipynb` |
|---|---|
| ⭐ 1 · ⭐⭐ 3 · ⭐⭐⭐ 4 · [ADV] · case study 1 | ⭐⭐ 2 · ⭐⭐⭐ 5 · case study 2 |

This notebook keeps its own save files (`labs/.state/s12-exercises-*`), deleted at the start, so your
`chat.py` tickets are never touched.

## Setup: given, just run it

In [ ]:
import json
import os
import sys
import time
import warnings
from pathlib import Path

from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", category=LangChainBetaWarning)
warnings.filterwarnings("ignore", message="IProgress not found")

HERE = Path.cwd()
STATE = HERE.parent / ".state"
for name, file in {"SHOPWISE_CHECKPOINT_DB": "s12-exercises-checkpoints.sqlite",
                   "SHOPWISE_MEMORY_DB": "s12-exercises-memory.sqlite",
                   "SHOPWISE_REFUND_LEDGER": "s12-exercises-refunds.jsonl"}.items():
    (STATE / file).unlink(missing_ok=True)
    os.environ[name] = str(STATE / file)
PROJECT = HERE / "project"
sys.path.insert(0, str(PROJECT))

from langchain.agents import create_agent                 # noqa: E402
from langchain_core.messages import HumanMessage          # noqa: E402
from langgraph.types import Command                       # noqa: E402

from chat import render                                   # noqa: E402
from core.config import settings                          # noqa: E402
from kb import rag, store                                 # noqa: E402
from middleware import Customer, context_for, desk_middleware   # noqa: E402
from models import build_model                            # noqa: E402
from workflows.events import run_config, stream_ticket    # noqa: E402
from workflows.specialists import mcp_client              # noqa: E402
from workflows.subagents import TEAM_TOOLS, answer_with_team    # noqa: E402
from workflows.supervisor import open_team                # noqa: E402

_team = open_team()
graph = _team.__enter__()
print(f"v{settings.VERSION} · {settings.model_id} · routing {settings.ROUTING}")

## ⭐ Exercise 1: Break the routing

**What this teaches:** in the subagents pattern, a wrapper's docstring **is** the routing.

Copy the team's tools, blur `ask_policy_expert`'s description to *"Answers questions."*, and run Tanvir's
policy question through the team with the blurred toolset (about 3–6 calls). Then compare with the real
docstring. Look at `consulted`, and at whether the reply still has a `[label]` citation.

In [ ]:
from workflows.subagents import ask_policy_expert

question = "How long is the warranty on a smartwatch?"
tanvir = context_for("tanvir@example.com")

blurred_policy = ask_policy_expert.model_copy(update={"description": "Answers questions."})
blurred = [blurred_policy if t.name == "ask_policy_expert" else t for t in TEAM_TOOLS]

for label, tools in (("real docstring", None), ("blurred", None)):   # TODO: pass `blurred` for the second
    out = answer_with_team(question, tanvir, tools=tools)
    print(f"{label}: consulted {out['consulted']}\n  cites: {sorted(rag.cited_sources(out['reply']))}\n  {out['reply'][:200]}\n")

**✍️ Your answer.** What changed in `consulted` and in the reply? In one sentence: why is a vague docstring a routing bug, not a style problem?

## ⭐⭐ Exercise 2: A new MCP tool *(MCP notebook)*

**What this teaches:** the orders specialist's tools arrive over the protocol, so a new server tool
reaches it with **no change to the agent code**.

The ShopWise server does not expose `list_orders`. Below, a small server file imports the real one and
adds a tool. Register `list_orders` (one line: `serve(list_orders)`), then:

1. list the server's tools (no model calls);
2. build the orders agent exactly as `workflows/specialists/orders.py` does, but from this server's
   tools, and ask *"What orders do I have?"* as Tanvir (about 2–3 calls).

In [ ]:
PLUS = STATE / "s12-server-plus.py"
PLUS.write_text(f"""
import sys
sys.path[:0] = [{str(PROJECT)!r}, {str(PROJECT / 'mcp_server')!r}]
from server import mcp, serve
from tools import list_orders

# TODO: register list_orders on the server, the same way server.py registers get_order

mcp.run()
""")

session = mcp_client.MCPSession({"transport": "stdio", "command": sys.executable, "args": [str(PLUS)],
                                 "cwd": str(PROJECT), "env": dict(os.environ)})
print([t.name for t in session.tools])
records = [session.sync_tool(t) for t in session.tools if t.name not in mcp_client.NOT_RECORDS]

from workflows.specialists.orders import PROMPTS
agent = create_agent(build_model(), tools=records,
                     system_prompt=PROMPTS["orders_prompt"].format(money=PROMPTS["orders_money_report"]),
                     middleware=desk_middleware(), context_schema=Customer)
tanvir = context_for("tanvir@example.com")
out = agent.invoke({"messages": [HumanMessage("Customer: Tanvir Ahmed, customer_id C-1002\nTask: What orders do I have?")]},
                   context=tanvir)
print([c["name"] for m in out["messages"] if m.type == "ai" for c in m.tool_calls])
print(out["messages"][-1].text)
session.close()

**✍️ Your answer.** Which lines of agent code did you change to give it the new tool? What did the docstring of `list_orders` have to say for the model to use it with `C-1002`?

## ⭐⭐ Exercise 3: Earn your keep

**What this teaches:** a technique ships only if the number moves on **this** corpus, and its cost is
counted where it is paid.

Score two-step RAG on the **plain** and **contextual** collections, on the five filtered questions plus
the four gold questions about PDFs and warranty (KBQ-10 to KBQ-15), about 2 × 11 calls. Then count the
contextual collection's ingest cost in model calls from `kb/contexts.json`.

In [ ]:
from evals.run_eval import questions, rag_eval_rows, score, table, two_step

rows = [q for q in questions() if q["id"] in {f"KBQ-{n}" for n in range(10, 16)}] + \
       [r for r in rag_eval_rows() if r["id"].startswith("KBS")]
results = {}   # TODO: {"plain": score(two_step(scheme="plain"), rows), "contextual": ...}
table(results)

contexts = json.loads((PROJECT / "kb/contexts.json").read_text())
print(f"\ncontextual ingest: {len(contexts)} chunks, so {len(contexts)} model calls, once")

**✍️ Your answer.** Keep it or drop it? Justify with the scorecard and the ingest cost, in three sentences. If nothing moved, what does `RAG-DECISIONS.md` say, and why is that a correct answer?

## ⭐⭐⭐ Exercise 4: Handoff with a summary, or with the transcript

**What this teaches:** what a handoff should carry, measured.

`transfer_to_refunds` carries a three-line summary. Write `transfer_with_transcript`, which carries the
orders agent's **whole** conversation instead (read it from `runtime.state["messages"]` through
`ToolRuntime`). Run three cancellation tickets through a supervisor-routed graph with each tool (about
6 × 5 calls), and compare: how big the handoff payload is (≈ characters ÷ 4 tokens), and whether the
refund proposal is the same.

In [ ]:
from langchain.tools import ToolRuntime, tool
from workflows.specialists import orders as orders_mod
from workflows.supervisor import build_team
from workflows.supervisor.handoffs import transfer_to_refunds

@tool
def transfer_with_transcript(order_id: str, runtime: ToolRuntime) -> Command:
    """Hand this customer to the refunds specialist: they want their money back for an order.
    Use it instead of answering when the customer asks to cancel for a refund. After calling it you are finished."""
    transcript = ""   # TODO: every message in runtime.state["messages"], one line each: "<type>: <text>"
    return Command(goto="refunds", graph=Command.PARENT,
                   update={"handoff": {"from": "orders", "order_id": order_id.strip().upper(), "summary": transcript},
                           "route_log": [{"by": "handoff", "to": "refunds", "from": "orders", "reason": "transcript"}]})

def run(tool, thread, text):
    orders_mod.orders_agent.cache_clear()
    original = orders_mod.transfer_to_refunds
    orders_mod.transfer_to_refunds = tool          # the agent is rebuilt with this handoff tool
    try:
        sup = build_team(checkpointer=graph.checkpointer, routing="supervisor")
        list(stream_ticket(sup, thread, text, context_for("tanvir@example.com")))
        state = sup.get_state(run_config(thread))
        handoff = next((e for e in state.values.get("route_log", []) if e["by"] == "handoff"), None)
        child = next((t.state.values for t in sup.get_state(run_config(thread), subgraphs=True).tasks if t.state), {})
        return {"handed_off": bool(handoff), "payload_tokens": len(child.get("reason", "")) // 4,
                "proposal": (child.get("proposal") or {}).get("recommendation")}
    finally:
        orders_mod.transfer_to_refunds = original
        orders_mod.orders_agent.cache_clear()

TICKETS = ["Is my watch ORD-5003 still in transit? If so, cancel it and refund me.",
           "ORD-5003 is taking forever. Where is it? I want to cancel and get my money back.",
           "Please check ORD-5003. If it hasn't been delivered, I'd like a refund instead."]
for n, text in enumerate(TICKETS):
    for name, tool in (("summary", transfer_to_refunds), ("transcript", transfer_with_transcript)):
        print(n, name, run(tool, f"EX4-{n}-{name}", text))

**✍️ Your answer.** Which wins on tokens, and did anything measurable get worse with the summary? In two sentences: when would a summary lose, and what would you do then?

## ⭐⭐⭐ Exercise 5: Two-way MCP *(MCP notebook)*

**What this teaches:** the assistant can be a **client of its own toolkit**.

Run a ticket through the team so it has a trace (LangSmith tracing must be on in `labs/.env`). Then build
an "ops" agent whose only tools are the server's `recent_runs` and `read_trace`, over MCP, and ask it as a
manager would: *"Why did you route EX5-ROUTE the way you did?"* (about 3–5 calls). Its answer should cite
the supervisor's `route_log`, not guess.

In [ ]:
from langchain_core.tracers.langchain import wait_for_all_tracers

render(stream_ticket(graph, "EX5-ROUTE", "Where is my watch (ORD-5003), and how long is its warranty?",
                     context_for("tanvir@example.com")))
wait_for_all_tracers()
time.sleep(10)                                   # LangSmith indexes a run a few seconds after it ends

trace_tools = []   # TODO: from mcp_client.server_tools(), keep only recent_runs and read_trace
ops = create_agent(build_model(), tools=trace_tools,
                   system_prompt="You explain ShopWise's past routing decisions to a manager, only from "
                                 "its traces. Find the run, read it, and quote the supervisor's route_log.")
out = ops.invoke({"messages": [HumanMessage("Why did you route EX5-ROUTE the way you did?")]})
print([c["name"] for m in out["messages"] if m.type == "ai" for c in m.tool_calls])
print(out["messages"][-1].text)

**✍️ Your answer.** Which tools did it call, and does the explanation match `graph.get_state(run_config('EX5-ROUTE')).values['route_log']`? Why does this agent need no authorize rail, and who should be allowed to use it?

## [ADV] The Market Analyst, and the Daily Brief part 2

Self-paced. Switch the fourth specialist on and answer opening A's question properly:

```python
os.environ["SHOPWISE_MARKET_ANALYST"] = "true"     # before building the graph; restart the kernel first
```

1. With `TAVILY_API_KEY` in `labs/.env` (free tier) or without it (DDGS, no key), run Kamal's ticket,
   *"Is your BrewMaster kettle overpriced, and can I return it if it is?"*, through `open_team()`. The
   supervisor should ask the market analyst first, then knowledge, with the price in the task.
2. Check the contract in the reply: a URL after every price, no invented price, weak sources flagged.
3. Daily Brief, part 2: add a **Market Watch** section to the report (`report/gather.py`) that calls
   `workflows.specialists.market.market_agent()` for each product low in stock. The other three sections
   stay deterministic: one graph, both paradigms.

## Case studies (no code)

**1. Org design is system design.** Map ShopWise's specialists to a real support org chart: the L1 desk,
the policy team, the finance approver, the market analyst. A new hire arrives: the **returns state
machine**, who walks a customer through identify → eligibility → label over several turns. Where does it
report, and which pattern is it?

**✍️ Your answer.**

**2. The MCP risk review** *(MCP notebook)*. A colleague wants to connect a popular third-party MCP server that reads email
to the support agent that can issue refunds. Write the three-line risk review (use the lethal trifecta),
and the configuration that would make it acceptable, if any.

**✍️ Your answer.**

In [ ]:
mcp_client.close()
_team.__exit__(None, None, None)
store.close_shared()